# B2.3 · Tool design

**Function B — Product & Application Security → The Security Automation / Harness Engineer**  ·  *Security of AI*

---

**Risk.** The dangerous call exists and is merely blocked.

**Control.** Read-only defaults, structured output contracts, allowlisted actions — design it out.

**This lab.** Refactor a shell tool into three narrow, structured tools.

| | |
|---|---|
| Open-source tooling | kmcp |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("B2.3"))

Tool design is security design. A tool's signature decides what the model is *able* to ask for, which is a stronger control than anything you can put in a prompt.

In [ ]:
from cybercommons import sandbox

guard = sandbox.PathGuard(workspace="/work")

# Bad tool: takes a free-form path. The model can ask for anything.
def read_file_bad(path):
    return guard.check(path)

# Good tool: takes an identifier the caller cannot use to escape.
FILES = {"app": "/work/src/app.py", "conf": "/work/conf.yaml"}
def read_file_good(name):
    path = FILES.get(name)
    if path is None:
        return sandbox.Decision(False, f"unknown file id (valid: {sorted(FILES)})", name)
    return guard.check(path)

for arg in ["/work/src/app.py", "/work/../../root/.ssh/id_rsa", "app", "conf"]:
    print(f"bad(\"{arg}\"): ", read_file_bad(arg))
print()
for arg in ["app", "conf", "/work/../../root/.ssh/id_rsa"]:
    print(f"good(\"{arg}\"):", read_file_good(arg))

Both tools are safe here because the guard runs underneath. The difference is what happens when the guard has a bug: the bad tool exposes it, the good tool never presents the surface at all.

### Expect

The free-form tool evaluates every path against the guard, including the traversal (denied). The identifier-based tool cannot express the traversal at all — it reports an unknown file id.

### Your turn

Rewrite one tool in your harness from free-form to enumerated. If you cannot, work out what the model genuinely needs the freedom for — that requirement is usually smaller than the current signature.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/B2.3.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*